# Extraction Results — Exact Match Evaluation

Micro-averaged Precision, Recall, F1 per category and overall.

**Micro-averaging** pools raw triple counts (matched, predicted, gold) before dividing — the correct approach when entries contain very few triples (1–3) and categories have different sizes.

In [21]:
import json
import pandas as pd
from collections import defaultdict
import re

RESULTS_PATH = "results/results_gemini_gemini-3-flash-preview.jsonl"

# ── Load results ──────────────────────────────────────────────────────────────
records = []
with open(RESULTS_PATH, encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

df = pd.DataFrame(records)
df["category"] = df["entry_id"].str.rsplit("_", n=2).str[0]
df["n_gold"] = df["gold_triples"].apply(len)
df["n_pred"] = df["pred_triples"].apply(len)

def add_underscores(text):
    """Replace any non-letter symbols that separate words with underscores."""
    return re.sub(r'[^a-zA-Z]+', '_', text).strip('_')

def process_triple(triple):
    """Add underscores to subject and object in a triple."""
    return {
        'subject': add_underscores(triple['subject']),
        'relation': triple['relation'],
        'object': add_underscores(triple['object'])
    }

# Apply to all gold triples
df['gold_triples'] = df['gold_triples'].apply(
    lambda triples: [process_triple(t) for t in triples]
)

# Reapply to predicted triples to use updated function
df['pred_triples'] = df['pred_triples'].apply(
    lambda triples: [process_triple(t) for t in triples]
)


print(f"Loaded {len(df)} entries across {df['category'].nunique()} categories")
print(f"Gold triples: {df['n_gold'].sum()}  |  Predicted triples: {df['n_pred'].sum()}")
df[["entry_id", "category", "n_gold", "n_pred"]].head(10)

Loaded 2103 entries across 57 categories
Gold triples: 4094  |  Predicted triples: 4100


,entry_id,category,n_gold,n_pred
0,1_Artist_test_1,1_Artist,1,1
1,1_Artist_test_2,1_Artist,1,1
2,1_Artist_test_3,1_Artist,1,1
3,1_Artist_test_4,1_Artist,1,1
4,1_Artist_test_5,1_Artist,1,1
5,1_Artist_test_6,1_Artist,1,1
6,1_Artist_test_7,1_Artist,1,1
7,1_Artist_test_8,1_Artist,1,1
8,1_Artist_test_9,1_Artist,1,1
9,1_Artist_test_10,1_Artist,1,1


In [22]:
# ── Exact-match helper ────────────────────────────────────────────────────────
def exact_match(pred: dict, gold: dict) -> bool:
    return (pred["subject"].lower() == gold["subject"].lower()
        and pred["relation"].lower() == gold["relation"].lower()
        and pred["object"].lower()  == gold["object"].lower())


def count_matched(pred_triples: list, gold_triples: list) -> int:
    """Count how many predicted triples have an exact match in gold (no double-counting)."""
    used = set()
    matched = 0
    for p in pred_triples:
        for j, g in enumerate(gold_triples):
            if j not in used and exact_match(p, g):
                matched += 1
                used.add(j)
                break
    return matched


df["matched"] = df.apply(lambda r: count_matched(r["pred_triples"], r["gold_triples"]), axis=1)

print(f"Total matched triples: {df['matched'].sum()} / {df['n_gold'].sum()} gold, {df['n_pred'].sum()} pred")

Total matched triples: 3072 / 4094 gold, 4100 pred


In [16]:
df.head(10)

,entry_id,input_text,gold_triples,pred_triples,pred_schemas,category,n_gold,n_pred,matched
0,1_Artist_test_1,Aaron Turner performs ambient music.,"[{'subject': 'Aaron_Turner', 'relation': 'genr...","[{'subject': 'Aaron_Turner', 'relation': 'genr...","[{'subject': 'MusicalArtist', 'object': 'Music...",1_Artist,1,1,1
1,1_Artist_test_2,Aaron Turner is an exponent of Doom metal.,"[{'subject': 'Aaron_Turner', 'relation': 'genr...","[{'subject': 'Aaron_Turner', 'relation': 'genr...","[{'subject': 'MusicalArtist', 'object': 'Music...",1_Artist,1,1,1
2,1_Artist_test_3,Aaron Turner played for Old Man Gloom.,"[{'subject': 'Aaron_Turner', 'relation': 'asso...","[{'subject': 'Aaron_Turner', 'relation': 'asso...","[{'subject': 'MusicalArtist', 'object': 'Band'}]",1_Artist,1,1,1
3,1_Artist_test_4,Aaron Turner originates from Boston.,"[{'subject': 'Aaron_Turner', 'relation': 'orig...","[{'subject': 'Aaron_Turner', 'relation': 'orig...","[{'subject': 'MusicalArtist', 'object': 'City'}]",1_Artist,1,1,1
4,1_Artist_test_5,Uruguay's leader is TabarГ© VГЎzquez.,"[{'subject': 'Uruguay', 'relation': 'leader', ...","[{'subject': 'Uruguay', 'relation': 'leader', ...","[{'subject': 'Country', 'object': 'Person'}]",1_Artist,1,1,1
5,1_Artist_test_6,Albennie Jones is a jazz artist.,"[{'subject': 'Albennie_Jones', 'relation': 'ge...","[{'subject': 'Albennie_Jones', 'relation': 'ge...","[{'subject': 'MusicalArtist', 'object': 'Music...",1_Artist,1,1,1
6,1_Artist_test_7,Hip Hop music has its stylistic origins in Jazz.,"[{'subject': 'Hip_hop_music', 'relation': 'sty...","[{'subject': 'Hip_Hop_music', 'relation': 'sty...","[{'subject': 'MusicGenre', 'object': 'MusicGen...",1_Artist,1,1,1
7,1_Artist_test_8,Aaron Turner falls in the genre of avant-garde...,"[{'subject': 'Aaron_Turner', 'relation': 'genr...","[{'subject': 'Aaron_Turner', 'relation': 'genr...","[{'subject': 'MusicalArtist', 'object': 'Music...",1_Artist,1,1,1
8,1_Artist_test_9,Ace Wilder's occupation is a singer.,"[{'subject': 'Ace_Wilder', 'relation': 'occupa...","[{'subject': 'Ace_Wilder', 'relation': 'occupa...","[{'subject': 'MusicalArtist', 'object': 'Profe...",1_Artist,1,1,1
9,1_Artist_test_10,The record label of Ace Wilder is Warner Music...,"[{'subject': 'Ace_Wilder', 'relation': 'record...","[{'subject': 'Ace_Wilder', 'relation': 'record...","[{'subject': 'MusicalArtist', 'object': 'Recor...",1_Artist,1,1,1


In [10]:
# ── Micro-averaged P / R / F1 per category ────────────────────────────────────
def micro_prf(group):
    m = group["matched"].sum()
    p_total = group["n_pred"].sum()
    g_total = group["n_gold"].sum()
    prec   = m / p_total if p_total else 0.0
    rec    = m / g_total if g_total else 0.0
    f1     = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    return pd.Series({
        "Precision": prec,
        "Recall":    rec,
        "F1":        f1,
        "Matched":   int(m),
        "Gold":      int(g_total),
        "Predicted": int(p_total),
        "Entries":   len(group),
    })


cat_df = df.groupby("category").apply(micro_prf, include_groups=False).sort_index()

# Format for display
styled = cat_df.style.format({
    "Precision": "{:.3f}", "Recall": "{:.3f}", "F1": "{:.3f}",
    "Matched": "{:.0f}", "Gold": "{:.0f}", "Predicted": "{:.0f}", "Entries": "{:.0f}",
})
styled

,Precision,Recall,F1,Matched,Gold,Predicted,Entries
category,,,,,,,
1_Airport,0.718,0.718,0.718,51,71,71,71
1_Artist,0.894,0.894,0.894,59,66,66,66
1_Astronaut,0.833,0.833,0.833,15,18,18,18
1_Athlete,0.731,0.742,0.737,49,66,67,66
1_Building,0.788,0.788,0.788,41,52,52,52
1_CelestialBody,0.825,0.825,0.825,33,40,40,40
1_City,0.180,0.180,0.180,11,61,61,61
1_ComicsCharacter,1.000,1.000,1.000,23,23,23,23
1_Company,0.909,0.909,0.909,20,22,22,22


In [30]:
# ── Overall micro-average ─────────────────────────────────────────────────────
overall = micro_prf(df)

print("=" * 50)
print("OVERALL (micro-averaged across all triples)")
print("=" * 50)
print(f"  Precision : {overall['Precision']:.4f}")
print(f"  Recall    : {overall['Recall']:.4f}")
print(f"  F1        : {overall['F1']:.4f}")
print(f"  Matched   : {int(overall['Matched'])} / {int(overall['Gold'])} gold, {int(overall['Predicted'])} predicted")
print(f"  Entries   : {int(overall['Entries'])}")

OVERALL (micro-averaged across all triples)
  Precision : 0.7493
  Recall    : 0.7504
  F1        : 0.7498
  Matched   : 3072 / 4094 gold, 4100 predicted
  Entries   : 2103


In [27]:
errors_inds = df[df["matched"] != df["n_gold"]].index.tolist()
def index_generator():
    for ind in errors_inds:
        yield ind
        
gen = index_generator()

In [31]:
idx = next(gen)
row = df.iloc[idx]

print("=" * 80)
print(f"Index: {idx} | Entry ID: {row['entry_id']}")
print(f"Category: {row['category']} | Matched: {row['matched']}/{row['n_gold']}")
print("-" * 80)
print(f"Input: {row['input_text']}")
print("-" * 80)
print("GOLD TRIPLES:")
for i, triple in enumerate(row['gold_triples'], 1):
    print(f"  {i}. ({triple['subject']}, {triple['relation']}, {triple['object']})")
print("-" * 80)
print("PRED TRIPLES:")
for i, triple in enumerate(row['pred_triples'], 1):
    print(f"  {i}. ({triple['subject']}, {triple['relation']}, {triple['object']})")
print("=" * 80)
print()

Index: 37 | Entry ID: 1_Artist_test_38
Category: 1_Artist | Matched: 0/1
--------------------------------------------------------------------------------
Input: Jazz originated its style from Blues music.
--------------------------------------------------------------------------------
GOLD TRIPLES:
  1. (Jazz, stylisticOrigin, Blues)
--------------------------------------------------------------------------------
PRED TRIPLES:
  1. (Jazz, stylisticOrigin, Blues_music)

